In [1]:
import pdfplumber
import pandas as pd
import numpy as np
import zipfile
import os
import re

### Load paths

In [108]:
# Paths (adjust as needed)
pdf_path = '../data/external/1-s2.0-S0956053X16300083-mmc2.pdf'  


In [97]:
#  Unzip the folder
#with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#    zip_ref.extractall(extract_dir)

### Functions

In [109]:
#Functions for data extractions
def extract_element(title, unit_text):
    # Example line: "C (wt%)"
    m = re.search(r'Value\s+ranges\s+for\s+(\w+)', title)
    element = m.group(1) if m else None
    u = re.search(r'\[([^\]]+)\]', unit_text)
    unit = u.group(1) if u else None
    
    return element, unit

def select_range(line_len):
    if line_len==18:
        return (4, 15)
    elif line_len==17:
        return (4, 14)
    elif line_len==20:
        return (4, 16)

def extract_fraction_data(lines, line_num):
    line = lines[line_num].strip()
    m = re.search(r'[\d\-]', line)  # Look for digit or dash
    if m:
        split_idx = m.start()
        fraction = line[:split_idx].strip()
        
        values_str = line[split_idx:]
        

        value_tokens = values_str.split(' ')
        values = [v if v != '-' else np.nan for v in value_tokens]
        return fraction, values

### Extract data from pdf

In [110]:
# Extract and parse page titles from PDF
page_data = {}
with pdfplumber.open(pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages, start=1):
        text = page.extract_text()
        if text:
            #Title is the first line and unit is the second line
            lines = text.split('\n')
            title = lines[0].strip() if lines else ''
            unit_text = lines[1].strip() if len(lines) > 1 else ''
            # Parse element and unit 
            element, unit = extract_element(title, unit_text)

            #Element index starts from 0 while pages number for elements starts with 1
            page_data[page_num-1] = {'element': element, 'unit': unit}
            
            range_tuple = select_range(len(lines)) 
            #select range based on number of lines, since Hg has an extra row of food waste (alternate calculation), and 
            #V missed the row for 'composites'
            for line_num in range(*range_tuple):  # lines 4-14
                if len(lines)!=20:
                #For all elements other than Hg, each fraction is extracted
                    fraction, values = extract_fraction_data(lines, line_num)
                    page_data[page_num-1][fraction] = values
                else:
                #For Hg, alterate calculation of food waste is discarded
                    fraction, values = extract_fraction_data(lines, line_num)
                    if values[0]!='-alt***':
                        page_data[page_num-1][fraction] = values
        else:
            print(f"Warning: No text on page {page_num}")



### Fix missing cells

In [111]:
#Some rows are missing value for the column 'n_<DL**', which is the second column. Insert np.nan for those rows to maintain consistency in the number of columns across all fractions.

for page_idx, data in page_data.items():
    # Extract fractions (exclude 'element' and 'unit')
    fractions_data = {k: v for k, v in data.items() if k not in ['element', 'unit']}
    for key, values in fractions_data.items():
        if len(values)!=9:
            print(f"Warning: Fraction '{key}' in Element '{data.get('element')}' has {len(values)} values instead of 9. Inserting np.nan at position 1 at column 'n_<DL**'")
            values.insert(1, np.nan)  # Insert np.nan at position 1
            print(f"After insertion, values are: {values}")
            

After insertion, values are: ['14', nan, '0.00', '0.00', '0.00', '0.20', '0.20', '0.20', '0.20']
After insertion, values are: ['12', nan, '0.32', '0.37', '0.95', '1.37', '1.73', '2.66', '2.97']
After insertion, values are: ['3', nan, '0.88', '0.88', '0.88', '0.96', '1.04', '1.04', '1.04']
After insertion, values are: ['32', nan, '3.90', '7.92', '9.57', '16.22', '28.86', '33.23', '70.00']
After insertion, values are: ['12', nan, '3.00', '3.54', '7.44', '13.03', '18.00', '23.08', '23.85']
After insertion, values are: ['2', nan, '42.00', '42.00', '42.00', '42.35', '42.70', '42.70', '42.70']
After insertion, values are: ['11', nan, '4.39', '4.69', '7.58', '10.20', '11.80', '17.52', '18.40']


### Organise data into dataframe

In [112]:
value_names = ['n_data*', 'n_<DL**', 'Min', '10%', '25%', 'Median', '75%', '90%', 'Max']  

dataframe = pd.DataFrame(columns = ['Element', 'Unit', 'Fraction']+value_names)

# For each page, create a DataFrame
for page_idx, data in page_data.items():
    # Extract fractions (exclude 'element' and 'unit')
    fractions_data = {k: v for k, v in data.items() if k not in ['element', 'unit']}
    
    # Create DataFrame with fractions as rows, value_names as columns
    df = pd.DataFrame(fractions_data, index=value_names).T
    df.reset_index(inplace=True)
    
    df.rename(columns={'index': 'Fraction'}, inplace=True)
    
    # Add element and unit as columns
    df['Element'] = data.get('element')
    df['Unit'] = data.get('unit')
    dataframe = pd.concat([dataframe, df], ignore_index=True)
    

In [ ]:
# Save or process the DataFrame (e.g., for page 0)
    #output_path = os.path.join(output_dir, f"page_{page_idx+1}_data.csv")
    #df.to_csv(output_path, index=False)
    #print(f"Saved DataFrame for page {page_idx+1}: {output_path}")

In [113]:
for col in value_names:
    for idx, val in dataframe[col].items():
        if isinstance(val, str) and '%' in val:
            val = val.replace('%', '')
        try:
            pd.to_numeric(val)
        except (ValueError, TypeError) as e:
            print(f"Conversion error in column '{col}', row {idx}: value='{val}' (error: {e})")

In [114]:
dataframe.to_csv('../data/processed/elemental_data_from_lit.csv', index=False)
